<a href="https://colab.research.google.com/github/estdanielscl/deeplearning/blob/main/proyectofinalcorte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn

IMPORTS

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

DATASET

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.GaussianBlur(3),
    transforms.ToTensor(),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_dataset = ImageFolder("data/train", transform=train_transforms)#datasetss
val_dataset = ImageFolder("data/val", transform=val_transforms)
test_dataset = ImageFolder("data/test", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

FileNotFoundError: [Errno 2] No such file or directory: 'data/train'

FOCAL LOSS

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss()

    def forward(self, outputs, targets):
        ce_loss = self.ce(outputs, targets)
        pt = torch.exp(-ce_loss)
        loss = (1 - pt) ** self.gamma * ce_loss
        return loss

MODELOS


4.1 Resnet50(base)

In [ ]:
class ResNet50Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet50(pretrained=True)

        for param in self.model.parameters():
            param.requires_grad = False

        self.model.fc = nn.Linear(self.model.fc.in_features, 2)

    def forward(self, x):
        return self.model(x)

4.2 EFFICIENT NET

In [ ]:
class EfficientNetModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.efficientnet_b0(pretrained=True)

        for param in self.model.parameters():
            param.requires_grad = False

        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, 2)

    def forward(self, x):
        return self.model(x)

CBAM

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        x = self.conv1(x)
        return self.sigmoid(x)

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.ca(x) * x # Aplica atención de canal
        x = self.sa(x) * x # Aplica atención espacial
        return x

FOURIER Y CNN

In [ ]:
def apply_fft(x):
    fft = torch.fft.fft2(x)
    fft = torch.fft.fftshift(fft)
    magnitud = torch.abs(fft)
    # +1 evita logaritmos de 0
    return torch.log(1 + magnitud)

In [ ]:
class FourierCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = apply_fft(x)
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

MODELO PROPUESTO CON FFT

In [ ]:
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Rama de Resnet
        resnet = models.resnet50(pretrained=True)
        for param in resnet.parameters():
            param.requires_grad = False
        # Da un mapa de características 2D (batch, 2048, 7, 7)
        self.resnet_features = nn.Sequential(*list(resnet.children())[:-2])
        # Se agrega CBAM
        self.cbam = CBAM(in_planes=2048)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # Rama de Fourier
        self.freq_conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),  # Normaliza los 16 canales
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),  # Normaliza los 32 canales
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.freq_fc = nn.Linear(32 * 56 * 56, 256)

        # Clasificacion final
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 256, 256),
            nn.BatchNorm1d(256), # Normaliza el vector de 256
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)
        )
    def forward(self, x):
        # Espacial
        spatial = self.resnet_features(x)  # Se extraen características en 2D
        spatial = self.cbam(spatial)       # Se activa CBAM
        spatial = self.avgpool(spatial)    # Aplana la imagen
        spatial = torch.flatten(spatial, 1)
        # Frecuencia
        freq = apply_fft(x)
        freq = self.freq_conv(freq)
        freq = freq.view(freq.size(0), -1)
        freq = self.freq_fc(freq)
        # Fusión de ambas ramas
        combined = torch.cat((spatial, freq), dim=1)
        return self.classifier(combined)


ENTRENAMIENTO DEL MODELO

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=30, patience=5):

    train_losses, val_losses, val_accuracies = [], [], []

    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_path = 'mejor_modelo_hibrido.pth'

    for epoch in range(num_epochs):
        # Entrenamiento
        model.train()
        running_loss = 0.0

        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)

        model.eval()
        val_running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = correct / total

        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        # Impresión de resultados
        print(f'Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

        # Early Stopping y guardado del mejor modelo obtenido
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"Nuevo mejor modelo guardado en: {best_model_path}")
        else:
            epochs_no_improve += 1
            print(f"Sin mejoras en validación por {epochs_no_improve} época(s).")
            if epochs_no_improve >= patience:
                print("Early stopping activado: Deteniendo el entrenamiento para evitar sobreajuste...")
                break # Termina el ciclo for inmediatamente
    # Cargar el mejor modelo obtenido
    print("\nCargando los pesos del mejor modelo obtenido durante el entrenamiento...")
    model.load_state_dict(torch.load(best_model_path))

    return model, train_losses, val_losses, val_accuracies

EVALUACION DEL MODELO

In [ ]:
def evaluate_model(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    preds, targets = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            preds.extend(predicted.cpu().numpy())
            targets.extend(labels.numpy())

    print(classification_report(targets, preds))

    cm = confusion_matrix(targets, preds)
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title("Confusion Matrix")
    plt.show()

GRAFICAS

In [ ]:
def plot_losses(train_losses, val_losses):
    plt.plot(train_losses, label="Train")
    plt.plot(val_losses, label="Validation")
    plt.legend()
    plt.title("Loss Curve")
    plt.show()

In [ ]:
model = HybridModel()

model, train_losses, val_losses, val_accuracies = train_model(model, train_loader, val_loader, criterion, optimizer)

plot_losses(train_losses, val_losses)

evaluate_model(model, test_loader)

Imagen original ──► ResNet50 ──► Features espaciales
        │
        ▼
FFT (Fourier) ──► CNN ──► Features de frecuencia

           ▼
      Concatenación

           ▼
   Clasificador final